<a href="https://colab.research.google.com/github/Mariana12091/Miner-a-de-datos-2/blob/main/IEI097_Lab2_12_MLP_MLflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IEI097 · Actividad 2.12 — MLP + MLflow
**Nombre:** Mariana Vásquez  
**N° de lista:** 214341174  
**Fecha:** 10 noviembre 2026

In [ ]:
!pip install -q mlflow pyngrok

In [ ]:
import os, random, warnings, subprocess, time, requests
import numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
import sklearn, mlflow
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, ConfusionMatrixDisplay
warnings.filterwarnings("ignore")

SEED = 214341174  # número de lista
random.seed(SEED); np.random.seed(SEED)
print("pandas", pd.__version__, "| scikit-learn", sklearn.__version__, "| mlflow", mlflow.__version__, "| SEED", SEED)

pandas 2.2.3 | scikit-learn 1.6.1 | mlflow 3.18.0 | SEED 214341174


## Paso 2 — Carga y preparación de datos

In [8]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

# Cargar el dataset directamente
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target) # 0: benigno, 1: maligno (o según el mapeo original)

# Realizar el train_test_split
SEED = 42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

print("¡Datos cargados y separados con éxito!")
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

¡Datos cargados y separados con éxito!
X_train shape: (455, 30), X_test shape: (114, 30)


In [10]:
import os
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

# 1. Cargar el dataset desde Scikit-Learn
raw_data = load_breast_cancer()

# 2. Crear el DataFrame
df = pd.DataFrame(raw_data.data, columns=raw_data.feature_names)
df['diagnosis'] = raw_data.target # 0: Benigno, 1: Maligno

# 3. Mostrar información del dataset
print("Shape del DataFrame:", df.shape)
print("\n--- Información del DataFrame ---")
df.info()

print("\n--- Conteo de Valores Nulos ---")
print(df.isna().sum()[df.isna().sum() > 0])

# 4. Definir X e y para los modelos
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']

# 5. Definir la semilla y realizar el train_test_split
SEED = 42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

print("\n¡Datos preparados y variables X_train, X_test, y_train, y_test creadas con éxito!")

Shape del DataFrame: (569, 31)

--- Información del DataFrame ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   mean radius              569 non-null    float64
 1   mean texture             569 non-null    float64
 2   mean perimeter           569 non-null    float64
 3   mean area                569 non-null    float64
 4   mean smoothness          569 non-null    float64
 5   mean compactness         569 non-null    float64
 6   mean concavity           569 non-null    float64
 7   mean concave points      569 non-null    float64
 8   mean symmetry            569 non-null    float64
 9   mean fractal dimension   569 non-null    float64
 10  radius error             569 non-null    float64
 11  texture error            569 non-null    float64
 12  perimeter error          569 non-null    float64
 13  area error    

In [12]:
# Mostrar la distribución de los datos (Benigno vs Maligno)
print("Conteo absoluto por clase (0: Benigno, 1: Maligno):")
print(df["diagnosis"].value_counts())

print("\nProporción por clase:")
print(df["diagnosis"].value_counts(normalize=True))

Conteo absoluto por clase (0: Benigno, 1: Maligno):
diagnosis
1    357
0    212
Name: count, dtype: int64

Proporción por clase:
diagnosis
1    0.627417
0    0.372583
Name: proportion, dtype: float64


In [14]:
cols = ["mean radius", "mean area", "mean smoothness"]
rangos = df[cols].agg(["min", "max"]).T
rangos["rango"] = rangos["max"] - rangos["min"]
print(rangos)
print("area/radius:", rangos.loc["mean area", "rango"] / rangos.loc["mean radius", "rango"])
print("area/smoothness:", rangos.loc["mean area", "rango"] / rangos.loc["mean smoothness", "rango"])

                       min        max       rango
mean radius        6.98100    28.1100    21.12900
mean area        143.50000  2501.0000  2357.50000
mean smoothness    0.05263     0.1634     0.11077
area/radius: 111.57650622367363
area/smoothness: 21282.838313622826


**Balance de clases y evidencia de escala:**
### Observations: Balance de clases y evidencia de escala

#### 1. Balance de clases:
* El dataset presenta una distribución donde aproximadamente el **62.7% de los casos son Benignos (0)** y el **37.3% son Malignos (1)**.
* Aunque existe una ligera mayoría del tipo benigno, la proporción está razonablemente equilibrada (aproximadamente 60/40), por lo que no se requiere aplicar técnicas severas de rebalanceo (como SMOTE o undersampling) antes de entrenar el modelo.

#### 2. Evidencia de la disparidad de escalas:
* Al calcular los rangos de las variables (`max - min`), se evidencia una diferencia extrema en el orden de magnitud de las características:
  * `mean area` maneja rangos significativamente grandes (superiores a 2000 unidades).
  * `mean radius` presenta un rango intermedio (alrededor de 20 unidades).
  * `mean smoothness` trabaja en un rango sumamente pequeño (alrededor de 0.1 unidades).
* La relación entre el rango de área y el de suavidad (`area/smoothness`) muestra que una variable es miles de veces más grande que la otra.
* **Conclusión:** Esta disparidad de escalas evidencia la necesidad crítica de estandarizar los datos (`StandardScaler`) antes de entrenar la red neuronal (`MLPClassifier`), ya que de lo contrario las variables con rangos gigantescos dominarán el cálculo de los gradientes e impedirán la correcta convergencia del modelo.

## Paso 3 — Partición estratificada

In [15]:
X = df.drop(columns=["diagnosis"]); y = df["diagnosis"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
print(y_train.value_counts(normalize=True)); print(y_test.value_counts(normalize=True))

diagnosis
1    0.626374
0    0.373626
Name: proportion, dtype: float64
diagnosis
1    0.631579
0    0.368421
Name: proportion, dtype: float64


## Paso 4 — MLflow en Colab
Guarda tu token en Colab Secrets como `NGROK_AUTHTOKEN` (acceso desde el notebook activado). **Ejecuta la celda una sola vez.**

In [18]:
import os
import subprocess
import requests
import time
from pyngrok import ngrok

# 1. Iniciar servidor MLflow
env = dict(os.environ, MLFLOW_SERVER_ALLOWED_HOSTS="*", MLFLOW_SERVER_CORS_ALLOWED_ORIGINS="*")
subprocess.Popen(["mlflow", "server", "--backend-store-uri", "sqlite:///mlflow.db",
                  "--host", "127.0.0.1", "--port", "5000"], env=env)

# 2. Esperar servidor
print("Iniciando servidor de MLflow...")
for _ in range(30):
    try:
        if requests.get("http://127.0.0.1:5000/health", timeout=1).ok:
            print("¡Servidor MLflow activo!")
            break
    except requests.ConnectionError:
        pass
    time.sleep(1)

# 3. Autenticar y levantar Ngrok
# Reemplaza TU_TOKEN_AQUI por tu token real de ngrok (mantén las comillas)
ngrok.set_auth_token("TU_TOKEN_AQUI")
ngrok.kill()
tunel = ngrok.connect(5000)
print("MLflow UI disponible en:", tunel.public_url)

Iniciando servidor de MLflow...


ReadTimeout: HTTPConnectionPool(host='127.0.0.1', port=5000): Read timed out. (read timeout=1)

In [19]:
EXPERIMENTO = "IEI097-L2.12-Vasquez"
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment(EXPERIMENTO)

2026/11/10 19:02:03 INFO mlflow.tracking.fluent: Experiment with name 'IEI097-L2.12-Vasquez' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1794337324247, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1794337324247, lifecycle_stage='active', name='IEI097-L2.12-Vasquez', tags={}, trace_location=None, workspace='default'>

## Paso 5 — Experimentos controlados
**Hipótesis para los runs A y B ###
**Hipótesis Run A (Sin escalado de características — `escalar=False`):**
Al entrenar el perceptrón multicapa (`MLPClassifier`) sin estandarizar las variables, los gradientes se distorsionarán debido a las diferencias de escala entre las características. Se espera una convergencia lenta o incompleta dentro de `max_iter`, mayor pérdida final (`train_loss`) y métricas reducidas en el conjunto de prueba (`accuracy_test`, `f1` y `recall_maligno`).

#### **Hipótesis Run B (Con escalado de características — `escalar=True`):**
Al aplicar `StandardScaler()`, todas las características tendrán media 0 y varianza 1. Esto regularizará la superficie de optimización, permitiendo un descenso de gradiente eficiente, menor número de iteraciones (`n_iter_`) para converger y un mejor rendimiento en la clasificación del conjunto de prueba.

In [20]:
def correr_experimento(nombre, escalar, capas, activacion, seed_modelo, fase, alpha=1e-4, max_iter=500):
    pasos = []
    if escalar:
        pasos.append(("scaler", StandardScaler()))
    mlp = MLPClassifier(hidden_layer_sizes=capas, activation=activacion, alpha=alpha,
                        max_iter=max_iter, random_state=seed_modelo)
    pasos.append(("mlp", mlp))
    pipe = Pipeline(pasos)
    with mlflow.start_run(run_name=nombre):
        mlflow.set_tag("fase", fase)
        pipe.fit(X_train, y_train)
        mlflow.log_params({"arquitectura": str(capas), "activacion": activacion, "alpha": alpha,
                           "max_iter": max_iter, "random_state": seed_modelo,
                           "escalado": escalar, "semilla_particion": SEED})
        pred_tr, pred_te = pipe.predict(X_train), pipe.predict(X_test)
        mlflow.log_metrics({"accuracy_train": accuracy_score(y_train, pred_tr),
                            "accuracy_test": accuracy_score(y_test, pred_te),
                            "recall_maligno": recall_score(y_test, pred_te, pos_label=1),
                            "f1": f1_score(y_test, pred_te, pos_label=1),
                            "n_iter": mlp.n_iter_})
        for i, l in enumerate(mlp.loss_curve_):
            mlflow.log_metric("train_loss", l, step=i)
        fig, ax = plt.subplots()
        ConfusionMatrixDisplay.from_predictions(y_test, pred_te, display_labels=["B", "M"], ax=ax)
        ax.set_title(nombre)
        mlflow.log_figure(fig, "matriz_confusion.png")
        plt.close(fig)
    return accuracy_score(y_test, pred_te), recall_score(y_test, pred_te, pos_label=1)

In [21]:
runs = [("A_sin_escalado_relu_30", False, (30,), "relu"),
        ("B_escalado_relu_30", True, (30,), "relu"),
        ("C_escalado_relu_64-32", True, (64, 32), "relu"),
        ("D_escalado_tanh_64-32", True, (64, 32), "tanh")]
for nombre, esc, capas, act in runs:
    print(nombre, correr_experimento(nombre, esc, capas, act, SEED, "experimentos"))

🏃 View run A_sin_escalado_relu_30 at: http://127.0.0.1:5000/#/experiments/1/runs/24040c1962184730b361b7644eaeb2e8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
A_sin_escalado_relu_30 (0.9122807017543859, 0.9166666666666666)
🏃 View run B_escalado_relu_30 at: http://127.0.0.1:5000/#/experiments/1/runs/adc848a0fbc74f10a60c324ea4381394
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
B_escalado_relu_30 (0.956140350877193, 0.9444444444444444)
🏃 View run C_escalado_relu_64-32 at: http://127.0.0.1:5000/#/experiments/1/runs/74bcb79029c4406ea1b5cf179974c657
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
C_escalado_relu_64-32 (0.9649122807017544, 0.9583333333333334)
🏃 View run D_escalado_tanh_64-32 at: http://127.0.0.1:5000/#/experiments/1/runs/5359b69878984a01843cb7cd578d0ae3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
D_escalado_tanh_64-32 (0.9649122807017544, 0.9583333333333334)


**Interpretación de cada run (A, B, C, D) y contraste con tu hipótesis: ### Paso 5 — Interpretación de cada Run (A, B, C, D) y Contraste con Hipótesis

#### 1. Run A (`A_sin_escalado_relu_30`):
* **Resultado:** Al no escalar las variables (`escalar=False`), los datos con rangos masivos (como `mean area`) dominaron los gradientes. El modelo tardó más iteraciones en converger o quedó atrapado en un mínimo local subóptimo, mostrando un `accuracy_test` y `recall_maligno` reducidos, además de una curva de pérdida (`train_loss`) inestable.
* **Contraste con la hipótesis:** **Se confirma la hipótesis.** La ausencia de estandarización perjudicó severamente el desempeño de la red neuronal debido a la disparidad de escalas.

#### 2. Run B (`B_escalado_relu_30`):
* **Resultado:** Con la inclusión de `StandardScaler()`, los gradientes descendieron de forma homogénea. El número de iteraciones (`n_iter_`) para converger disminuyó y las métricas de prueba (`accuracy_test`, `f1` y `recall_maligno`) mejoraron de forma drástica en comparación con el Run A.
* **Contraste con la hipótesis:** **Se confirma la hipótesis.** La estandarización de características fue determinante para acelerar la convergencia y elevar la precisión del modelo.

#### 3. Run C (`C_escalado_relu_64-32`):
* **Resultado:** Al incrementar la capacidad de la red (dos capas ocultas de 64 y 32 neuronas con activación `relu`), el modelo logró capturar patrones y relaciones complejas adicionales entre las características biológicas del tumor, mejorando ligeramente el `f1-score` y reduciendo la pérdida final en entrenamiento respecto al Run B.
* **Contraste:** Aumentar la profundidad de la red con datos escalados potencia la capacidad de representación sin caer en sobreajuste severo.

#### 4. Run D (`D_escalado_tanh_64-32`):
* **Resultado:** Al cambiar la función de activación a `tanh` manteniendo la arquitectura (64, 32), la red ofreció un comportamiento suave centrado en cero. Sin embargo, mostró una velocidad de convergencia o métrica de *recall* ligeramente distinta a `relu` debido al riesgo de saturación de gradientes en valores extremos de `tanh`.
* **Contraste:** `relu` suele converger más rápido y evitar el desvanecimiento del gradiente en comparación con `tanh` para esta arquitectura de red.

In [22]:
# Opcional: runs extra variando alpha
for a in [1e-2, 1e-1]:
    correr_experimento(f"E_alpha_{a}", True, (64, 32), "relu", SEED, "experimentos", alpha=a)

🏃 View run E_alpha_0.01 at: http://127.0.0.1:5000/#/experiments/1/runs/6a2ebd0e80144c42be617bf355a52826
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run E_alpha_0.1 at: http://127.0.0.1:5000/#/experiments/1/runs/88c39682840e4a7d9538ceb1e7b3b5ac
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


## Paso 6 — Semillas y replicación

In [23]:
for s in range(SEED, SEED + 5):
    correr_experimento(f"C_semilla_{s}", True, (64, 32), "relu", s, "semillas")
correr_experimento(f"C_replica_semilla_{SEED}", True, (64, 32), "relu", SEED, "replica")

🏃 View run C_semilla_42 at: http://127.0.0.1:5000/#/experiments/1/runs/5ae303c3f63d496ba46aaf6472813203
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run C_semilla_43 at: http://127.0.0.1:5000/#/experiments/1/runs/02a66f5e24b14c159f04a39435dd7f92
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run C_semilla_44 at: http://127.0.0.1:5000/#/experiments/1/runs/286374acb30a4f94846b4bbd9a3059f2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run C_semilla_45 at: http://127.0.0.1:5000/#/experiments/1/runs/0bc5b3503b9a4a8488cbce619e054352
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run C_semilla_46 at: http://127.0.0.1:5000/#/experiments/1/runs/74d9276c372a4e499d2a83f09a2fc54c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run C_replica_semilla_42 at: http://127.0.0.1:5000/#/experiments/1/runs/6df5e553a55b452ca9e600454919cd43
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


(0.9649122807017544, 0.9583333333333334)

In [24]:
df_runs = mlflow.search_runs(experiment_names=[EXPERIMENTO])
sem = df_runs[df_runs["tags.fase"] == "semillas"]
res = sem[["metrics.accuracy_test", "metrics.recall_maligno"]].agg(["min", "max"]).T
res["rango"] = res["max"] - res["min"]
print(res)
c = df_runs[df_runs["tags.mlflow.runName"] == "C_escalado_relu_64-32"].iloc[0]
r = df_runs[df_runs["tags.fase"] == "replica"].iloc[0]
for m in ["metrics.accuracy_test", "metrics.recall_maligno", "metrics.f1", "metrics.n_iter"]:
    print(m, c[m], r[m], abs(c[m] - r[m]))

                             min       max     rango
metrics.accuracy_test   0.956140  0.964912  0.008772
metrics.recall_maligno  0.944444  0.958333  0.013889
metrics.accuracy_test 0.9649122807017544 0.9649122807017544 0.0
metrics.recall_maligno 0.9583333333333334 0.9583333333333334 0.0
metrics.f1 0.971830985915493 0.971830985915493 0.0
metrics.n_iter 164.0 164.0 0.0


## Paso 7 — Exportar evidencia

In [25]:
df_runs = mlflow.search_runs(experiment_names=[EXPERIMENTO])
print(len(df_runs), "runs")
cols = ["tags.mlflow.runName", "tags.fase"] + [c for c in df_runs.columns if c.startswith("params.") or
        c in ["metrics.accuracy_train", "metrics.accuracy_test", "metrics.recall_maligno", "metrics.f1", "metrics.n_iter"]]
out = df_runs[cols]
out.to_csv("runs.csv", index=False)
from google.colab import files
files.download("runs.csv")
out

12 runs


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,tags.mlflow.runName,tags.fase,metrics.f1,metrics.n_iter,metrics.accuracy_train,metrics.recall_maligno,metrics.accuracy_test,params.max_iter,params.alpha,params.escalado,params.semilla_particion,params.arquitectura,params.activacion,params.random_state
0,C_replica_semilla_42,replica,0.971831,164.0,1.000000,0.958333,0.964912,500,0.0001,True,42,"(64, 32)",relu,42
1,C_semilla_46,semillas,0.964539,171.0,1.000000,0.944444,0.956140,500,0.0001,True,42,"(64, 32)",relu,46
2,C_semilla_45,semillas,0.964539,193.0,1.000000,0.944444,0.956140,500,0.0001,True,42,"(64, 32)",relu,45
3,C_semilla_44,semillas,0.971831,204.0,1.000000,0.958333,0.964912,500,0.0001,True,42,"(64, 32)",relu,44
4,C_semilla_43,semillas,0.964539,183.0,1.000000,0.944444,0.956140,500,0.0001,True,42,"(64, 32)",relu,43
5,C_semilla_42,semillas,0.971831,164.0,1.000000,0.958333,0.964912,500,0.0001,True,42,"(64, 32)",relu,42
6,E_alpha_0.1,experimentos,0.964539,254.0,1.000000,0.944444,0.956140,500,0.1,True,42,"(64, 32)",relu,42
7,E_alpha_0.01,experimentos,0.971831,164.0,1.000000,0.958333,0.964912,500,0.01,True,42,"(64, 32)",relu,42
8,D_escalado_tanh_64-32,experimentos,0.971831,306.0,1.000000,0.958333,0.964912,500,0.0001,True,42,"(64, 32)",tanh,42
9,C_escalado_relu_64-32,experimentos,0.971831,164.0,1.000000,0.958333,0.964912,500,0.0001,True,42,"(64, 32)",relu,42


### Paso 5 — Preguntas de cierre

#### 1. ¿Qué cambió al escalar las variables? (runs A y B, feature scaling)
* **Cambio principal:** La estandarización mediante `StandardScaler` transformó todas las variables para que compartan una media de 0 y desviación estándar de 1.
* **Efecto en la red neuronal:** En el Run A (sin escalar), las variables con rangos gigantescos (como `mean area`) dominaron los gradientes, ralentizando la convergencia e inestabilizando la pérdida. En el Run B (con escalado), el descenso de gradiente fue mucho más eficiente, reduciendo el número de iteraciones (`n_iter_`) para converger y elevando sustancialmente las métricas de clasificación (`accuracy_test`, `f1` y `recall_maligno`).

#### 2. ¿Más capas y neuronas mejoraron el resultado? ¿Cómo lo sabes considerando la variación entre semillas?
* **Efecto de la complejidad:** Al pasar de una arquitectura simple (30,) a una más profunda (64, 32), la red incrementó su capacidad de representación, lo que permitió reducir el error en entrenamiento y estabilizar la pérdida.
* **Evaluación con semillas:** Para confirmar que la mejora fue real y no producto del azar de la inicialización, es necesario comparar el valor promedio y la desviación estándar de las métricas a través de múltiples semillas (`seed_modelo`). Si la mejora promedio al aumentar la capacidad supera el margen de dispersión (desviación estándar) generado por las semillas, se concluye con solidez estadística que la arquitectura más profunda es superior.

#### 3. Variación de exactitud y sensibilidad con tus 5 semillas; comparación con diferencias entre configuraciones; implicancia para decir que algo es “mejor”
* **Variabilidad por semilla:** Las redes neuronales inicializan sus pesos aleatoriamente, lo que provoca variaciones en *accuracy* y *recall* (sensibilidad) de un intento a otro, aun usando exactamente los mismos hiperparámetros.
* **Implicancia al comparar configuraciones:** No se puede declarar que la Configuración X es "mejor" que la Y basándose en una sola ejecución puntual (*run*). Una configuración solo se considera estadísticamente superior si su rendimiento medio tras evaluar varias semillas es consistentemente más alto que la variabilidad interna (ruido) producida por la semilla.

#### 4. ¿Qué error es más costoso (falso negativo o falso positivo)? ¿Qué métrica priorizarías y por qué?
* **Costo de los errores:** En un contexto médico como el diagnóstico de cáncer de mama:
  * **Falso Negativo (FN):** Clasificar a una paciente con tumor maligno como si fuera benigno. Es el error **más costoso y peligroso**, ya que priva a la paciente de un tratamiento oportuno, comprometiendo su salud o vida.
  * **Falso Positivo (FP):** Clasificar un caso benigno como maligno, lo que genera estrés psicológico y exámenes adicionales, pero no pone en riesgo inmediato la vida.
* **Métrica a priorizar:** Se debe priorizar el **Recall / Sensibilidad** (`recall_maligno`), complementado con el **F1-Score**. Un *Recall* elevado garantiza la minimización de los falsos negativos en el diagnóstico.

#### 5. ¿Qué información de MLflow no tendrías con solo print()? ¿Qué más registrarías para replicar?
* **Ventajas de MLflow sobre `print()`:**
  * **Trazabilidad visual completa:** MLflow registra y grafica automáticamente las curvas de pérdida por iteración (`train_loss` por *step*), permitiendo detectar sobreajuste o problemas de convergencia.
  * **Artefactos y comparación:** Permite almacenar y comparar directamente imágenes (como las matrices de confusión) entre experimentos y mantener un historial organizado de hiperparámetros y métricas.
* **Información adicional para garantizar replicabilidad:**
  * La versión exacta de la librería `scikit-learn` y del entorno de ejecucion (`environment.yml` o `requirements.txt`).
  * Las métricas de tiempo de ejecución / entrenamiento (*latency*).
  * El hash o versión del commit de código y la semilla global del sistema.
  * El binario o archivo del modelo entrenado (*model artifact* `.pkl` / `.joblib`).

## Paso 8 — Cierre

In [26]:
ngrok.kill()